### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing.LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set field validation, descriptions, and nested structures.

In [1]:

from os import getenv
import os
from langchain.chat_models import init_chat_model
os.environ['GROQ_API_KEY']=getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000019D3F878CD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000019D3F9D5590>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    ratings:float=Field(description="The movie's rating out of 10")
    

In [3]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000019D3F878CD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000019D3F9D5590>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title':

In [4]:
response=model_with_structure.invoke("Provide details of the movie RRR")
response

Movie(title='RRR', year=2022, director='S.S. Rajamouli', ratings=8.8)

In [5]:
model.invoke("Provide details of the movie RRR")

AIMessage(content='<think>\nOkay, the user is asking for details about the movie RRR. Let me start by recalling what I know about it. RRR is a 2022 Indian film directed by S.S. Rajamouli. It\'s a big budget movie, which I think is part of the reason it gained so much attention globally. The title stands for "Rise, Roar, Revolt," which I believe is a play on the phrase "Rise and Roar" but with an extra R. \n\nThe main cast includes Ram Charan and Jr. NTR, who play two fictional revolutionaries based on real historical figures. I think their characters are inspired by Alluri Sitarama Raju and Komaram Bheem. The movie is set during the Indian independence movement, so that\'s the historical context. \n\nThe user might want to know the plot, so I should explain the story. The film follows two freedom fighters from different backgrounds who join forces to resist British colonial rule. There\'s a love story element too, which is quite prominent in the movie. The female lead is Alia Bhatt, wh

### Message Output alongside parsed structure

In [ ]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """The movie Details"""
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="This year the movie was released")
    director:str=Field(...,description="The director of the movie")
    ratings:float=Field(...,description="The movie's rating out of 10")

model_with_structure=model.with_structured_output(Movie,include_raw=True)

response= model_with_structure.invoke("Provide details of the movie RRR")
response



{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie RRR. Let me check what functions I have available. There\'s a Movie function that requires title, year, director, and ratings. I need to make sure I have all that information for RRR.\n\nFirst, the title is obviously "RRR". The year it was released—I think it came out in 2022. The director is SS Rajamouli, right? He\'s known for big budget films. As for ratings, I recall that RRR has a high rating on IMDb, maybe around 8.0 or higher. Let me confirm those details. \n\nWait, I should be accurate. Let me double-check the release year. Yes, 2022. The director is definitely SS Rajamouli. The ratings—IMDb lists it at 8.1. Okay, that\'s solid. So I can plug these into the Movie function parameters. Make sure all required fields are included: title, year, director, ratings. No optional parameters here. \n\nI need to structure the JSON correctly. The function name is "Movi

### Nested Structure

In [7]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(
        None,
        description="Budget in millions USD"
    )


model_with_structure=model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("Provide details of the movie RRR")
response

MovieDetails(title='RRR', year=2022, cast=[Actor(name='Ram Charan', role='Rajam'), Actor(name='N. T. Rama Rao Jr.', role='Raju'), Actor(name='Pooja Hegde', role='Mandovi')], genres=['Action', 'Drama'], budget=None)


### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [8]:

from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [9]:

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Science Fiction', 'Adventure'],
 'title': 'Avengers',
 'year': 2012}

In [10]:

model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [13]:
import os
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [12]:
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field
from operator import truth
from langchain.agents import create_agent

# Load environment variables from .env file
load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


# Use the Google GenAI model via the correct 'google_genai:' provider prefix
agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=ContactInfo
)


result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='7ef63771-d238-42b1-afc0-0aee42332e64'),
  AIMessage(content='{\n  "name": "John Doe",\n  "email": "john@example.com",\n  "phone": "(555) 123-4567"\n}', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f112c-fed3-7be1-8e05-dd01340180e3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 29, 'output_tokens': 44, 'total_tokens': 73, 'input_token_details': {'cache_read': 0}})],
 'structured_response': ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')}

In [15]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [16]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')